# 11 — Lasso Regression: Final Model

### How we got here

We tested **8 linear regression configurations** across 5 positions (nb09), varying features (delta vs. absolute) and targets (delta vs. post). The best config was **M4b** — using origin team projected style, destination team current style, and pre-transfer player qualities to predict post-transfer player qualities (R²=0.356).

Then (nb10) we added engineered features (style distance, pre-minutes) and compared model families:

| Model | Mean R² | Takeaway |
|-------|---------|----------|
| OLS baseline | 0.356 | Decent starting point |
| OLS + features | 0.355 | New features didn’t help linearly |
| Ridge | 0.366 | Regularization helps |
| **Lasso** | **0.369** | **Best — auto feature selection** |
| XGBoost | 0.282 | Overfits with limited data (~600-1500 rows/position) |

**Key learnings:**
- Pre-transfer player qualities are the single most important predictor (R² jumps from 0.01 to 0.27 when added)
- Regularization matters more than model complexity
- The problem is fundamentally linear at this data scale

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LassoCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

DATA_ROOT = '/Users/jorgepadilla/Documents/Documents - Jorge\u2019s MacBook Air/thesis_data'
V1 = f'{DATA_ROOT}/processed_data/model_dataset/v_1'
raw = pd.read_parquet(f'{V1}/model_df.parquet')

## 1. Data Prep

In [2]:
df = raw[raw['position'] != 'Goalkeeper'].copy()
team_cols = [c for c in df.columns if c.startswith('from_q_proj_') or c.startswith('to_q_') or c.startswith('delta_team_')]
df = df[~df[team_cols].isna().any(axis=1)].copy()

ALL_PLAYER_QUALITIES = [
    'Active defence', 'Aerial threat', 'Box threat', 'Chance prevention',
    'Composure', 'Defensive heading', 'Dribbling', 'Effectiveness',
    'Finishing', 'Hold-up play', 'Intelligent defence', 'Involvement',
    'Passing quality', 'Poaching', 'Pressing', 'Progression',
    'Providing teammates', 'Run quality', 'Territorial dominance', 'Winning duels',
]
TEAM_ALL = ['DEFENCE', 'DEFENSIVE_TRANSITION', 'ATTACKING_TRANSITION',
            'ATTACK', 'PENETRATION', 'CHANCE_CREATION', 'OUTCOME']
POSITIONS = sorted(df['position'].unique())

# Per-position qualities
position_qualities = {}
for pos in POSITIONS:
    mask = df['position'] == pos
    position_qualities[pos] = [q for q in ALL_PLAYER_QUALITIES if df.loc[mask, f'pre_{q}'].isna().mean() < 0.50]

# Style distance
from_cols = [f'from_q_proj_{q}' for q in TEAM_ALL]
to_cols = [f'to_q_{q}' for q in TEAM_ALL]
df['style_distance'] = np.sqrt(((df[to_cols].values - df[from_cols].values) ** 2).sum(axis=1))

# Train/test
train_mask = df['to_season'].isin([2020, 2021, 2022, 2023])
test_mask = df['to_season'] == 2024

print(f'{len(df):,} transfers | Train: {train_mask.sum():,} | Test: {test_mask.sum():,}')

6,677 transfers | Train: 5,387 | Test: 1,290


## 2. Fit Lasso per Position

In [3]:
NEW_FEATURES = ['style_distance', 'pre_minutes']

def get_feature_cols(qualities):
    from_proj = [f'from_q_proj_{q}' for q in TEAM_ALL]
    to_curr = [f'to_q_{q}' for q in TEAM_ALL]
    pre = [f'pre_{q}' for q in qualities]
    return from_proj + to_curr + pre + NEW_FEATURES

# Store fitted models and results
models = {}
all_results = []

for pos in POSITIONS:
    qualities = position_qualities[pos]
    x_cols = get_feature_cols(qualities)
    y_cols = [f'post_{q}' for q in qualities]

    pos_df = df[df['position'] == pos]
    clean = pos_df[x_cols + y_cols].dropna()
    train_idx = clean.index[clean.index.isin(df[train_mask].index)]
    test_idx = clean.index[clean.index.isin(df[test_mask].index)]

    if len(train_idx) < 20 or len(test_idx) < 5:
        print(f'  SKIP {pos}: train={len(train_idx)}, test={len(test_idx)}')
        continue

    X_train = clean.loc[train_idx, x_cols].values
    X_test = clean.loc[test_idx, x_cols].values
    Y_train = clean.loc[train_idx, y_cols].values
    Y_test = clean.loc[test_idx, y_cols].values

    model = MultiOutputRegressor(LassoCV(alphas=np.logspace(-3, 1, 20), max_iter=5000))
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    models[pos] = {
        'model': model, 'x_cols': x_cols, 'y_cols': y_cols,
        'qualities': qualities, 'n_train': len(train_idx), 'n_test': len(test_idx),
    }

    for i, q in enumerate(qualities):
        all_results.append({
            'position': pos, 'quality': q,
            'r2': r2_score(Y_test[:, i], Y_pred[:, i]),
            'mae': mean_absolute_error(Y_test[:, i], Y_pred[:, i]),
        })

    mean_r2 = np.mean([r['r2'] for r in all_results if r['position'] == pos])
    print(f'  {pos:20s}: {len(train_idx):4d} train, {len(test_idx):3d} test, '
          f'{len(x_cols)} features -> {len(y_cols)} targets | mean R\u00b2={mean_r2:.3f}')

results_df = pd.DataFrame(all_results)

  SKIP Central Defender: train=316, test=0
  Full Back           :  652 train, 192 test, 36 features -> 20 targets | mean R²=0.301
  Midfielder          : 1535 train, 383 test, 33 features -> 17 targets | mean R²=0.468
  Striker             :  576 train, 117 test, 34 features -> 18 targets | mean R²=0.388
  Winger              :  763 train, 189 test, 34 features -> 18 targets | mean R²=0.319


## 3. Feature Selection — What Lasso Kept

Lasso automatically sets irrelevant feature weights to exactly zero. Below we see which features survived per position.

In [4]:
# Per position: count non-zero coefficients per feature (across all target qualities)
for pos in models:
    info = models[pos]
    coefs = np.array([est.coef_ for est in info['model'].estimators_])  # (n_targets, n_features)
    n_targets = coefs.shape[0]
    n_nonzero = (coefs != 0).sum(axis=0)  # how many targets use each feature

    feat_usage = pd.Series(n_nonzero, index=info['x_cols'])
    kept = (feat_usage > 0).sum()
    dropped = (feat_usage == 0).sum()
    print(f'{pos}: {kept}/{len(info["x_cols"])} features kept, {dropped} fully zeroed out')
    if dropped > 0:
        zeroed = feat_usage[feat_usage == 0].index.tolist()
        short_names = [n.replace('from_q_proj_', 'from_').replace('to_q_', 'to_').replace('pre_', 'pre_') for n in zeroed]
        print(f'  Dropped: {short_names}')

Full Back: 36/36 features kept, 0 fully zeroed out
Midfielder: 33/33 features kept, 0 fully zeroed out
Striker: 34/34 features kept, 0 fully zeroed out
Winger: 34/34 features kept, 0 fully zeroed out


In [5]:
# Coefficient heatmap for each position
for pos in models:
    info = models[pos]
    coefs = np.array([est.coef_ for est in info['model'].estimators_])
    coef_df = pd.DataFrame(coefs, index=info['qualities'], columns=info['x_cols'])

    # Shorten names
    short_cols = (
        coef_df.columns
        .str.replace('from_q_proj_', 'from ', regex=False)
        .str.replace('to_q_', 'to ', regex=False)
        .str.replace('pre_', 'pre ', regex=False)
    )
    coef_df.columns = short_cols

    # Only show features that have at least one non-zero coef
    active_mask = (coef_df != 0).any(axis=0)
    coef_active = coef_df.loc[:, active_mask]

    fig = px.imshow(
        coef_active.round(3),
        text_auto='.2f',
        color_continuous_scale='RdBu_r',
        zmin=-0.6, zmax=0.6,
        labels=dict(x='Feature', y='Target Quality', color='Coef'),
        aspect='auto',
    )
    fig.update_layout(
        title=f'{pos} — Lasso Coefficients (non-zero only)',
        height=max(400, len(info['qualities']) * 28),
        width=max(600, coef_active.shape[1] * 38),
        margin=dict(t=50, b=60, l=160, r=30),
        xaxis_tickangle=-45,
    )
    fig.show()

## The Model

For each position $p$ and each target quality $j$, Lasso solves:

$$\hat{q}_{j}^{\,\text{post}} = \beta_0^{(j)} + \sum_{k=1}^{7} \alpha_k^{(j)} \cdot S_k^{\,\text{from}} + \sum_{k=1}^{7} \gamma_k^{(j)} \cdot S_k^{\,\text{to}} + \sum_{i=1}^{N_p} \omega_i^{(j)} \cdot q_i^{\,\text{pre}} + \phi^{(j)} \cdot d_{\text{style}} + \psi^{(j)} \cdot m_{\text{pre}}$$

subject to the L1 penalty:

$$\min_{\boldsymbol{\beta}} \; \frac{1}{2n} \| \mathbf{y}_j - \mathbf{X}\boldsymbol{\beta}_j \|_2^2 \;+\; \lambda \|\boldsymbol{\beta}_j\|_1$$

where:

| Symbol | Meaning |
|--------|---------|
| $S_k^{\,\text{from}},\; S_k^{\,\text{to}}$ | Origin / destination team style qualities (7 each: Defence, Def. Transition, Att. Transition, Attack, Penetration, Chance Creation, Outcome) |
| $q_i^{\,\text{pre}}$ | Player quality $i$ before transfer ($N_p$ = 17–20 depending on position) |
| $d_{\text{style}}$ | Style distance: $\sqrt{\sum_k (S_k^{\,\text{to}} - S_k^{\,\text{from}})^2}$ |
| $m_{\text{pre}}$ | Minutes played at origin team |
| $\lambda$ | Regularization strength (selected via cross-validation) |
| $\|\boldsymbol{\beta}_j\|_1$ | L1 penalty — drives irrelevant coefficients to exactly zero |

## 4. Model Performance

In [6]:
# R² per quality x position heatmap
pivot_r2 = results_df.pivot_table(index='quality', columns='position', values='r2')

fig = px.imshow(
    pivot_r2.round(3),
    text_auto='.3f',
    color_continuous_scale='RdYlGn',
    zmin=-0.3, zmax=0.8,
    labels=dict(x='Position', y='Quality', color='R\u00b2'),
    aspect='auto',
)
fig.update_layout(
    title='R\u00b2 per Quality x Position (test set: 2024)',
    height=600, width=650,
    margin=dict(t=50, b=30, l=170, r=30),
)
fig.show()

In [7]:
# Mean R² and MAE by position
pos_summary = results_df.groupby('position').agg(
    mean_r2=('r2', 'mean'),
    mean_mae=('mae', 'mean'),
    median_r2=('r2', 'median'),
).round(4)

for pos in models:
    pos_summary.loc[pos, 'n_train'] = models[pos]['n_train']
    pos_summary.loc[pos, 'n_test'] = models[pos]['n_test']
    pos_summary.loc[pos, 'n_features'] = len(models[pos]['x_cols'])
    pos_summary.loc[pos, 'n_targets'] = len(models[pos]['qualities'])

pos_summary[['n_train', 'n_test', 'n_features', 'n_targets']] = pos_summary[['n_train', 'n_test', 'n_features', 'n_targets']].astype(int)
print(pos_summary.to_string())

# Overall
print(f'\nOverall mean R\u00b2: {results_df["r2"].mean():.4f}')
print(f'Overall mean MAE: {results_df["mae"].mean():.4f}')

            mean_r2  mean_mae  median_r2  n_train  n_test  n_features  n_targets
position                                                                        
Full Back    0.3008    0.4235     0.2961      652     192          36         20
Midfielder   0.4680    0.3770     0.4770     1535     383          33         17
Striker      0.3883    0.3875     0.4360      576     117          34         18
Winger       0.3191    0.4190     0.2705      763     189          34         18

Overall mean R²: 0.3658
Overall mean MAE: 0.4027


In [8]:
# Bar chart: mean R² by position
fig = make_subplots(rows=1, cols=2, subplot_titles=['Mean R\u00b2 by Position', 'Mean MAE by Position'])

colors = px.colors.qualitative.Set2
positions_sorted = pos_summary.sort_values('mean_r2', ascending=False).index.tolist()

fig.add_trace(go.Bar(
    x=positions_sorted,
    y=[pos_summary.loc[p, 'mean_r2'] for p in positions_sorted],
    marker_color=colors[:len(positions_sorted)],
    text=[f'{pos_summary.loc[p, "mean_r2"]:.3f}' for p in positions_sorted],
    textposition='outside', showlegend=False,
), row=1, col=1)

fig.add_trace(go.Bar(
    x=positions_sorted,
    y=[pos_summary.loc[p, 'mean_mae'] for p in positions_sorted],
    marker_color=colors[:len(positions_sorted)],
    text=[f'{pos_summary.loc[p, "mean_mae"]:.3f}' for p in positions_sorted],
    textposition='outside', showlegend=False,
), row=1, col=2)

fig.update_layout(height=400, width=800, margin=dict(t=50, b=30, l=40, r=20))
fig.show()

In [9]:
# R² distribution: box plot per position
fig = px.box(
    results_df, x='position', y='r2', color='position',
    points='all',
    labels={'r2': 'R\u00b2', 'position': 'Position'},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(
    title='R\u00b2 Distribution per Position (each dot = one player quality)',
    height=400, width=700,
    margin=dict(t=50, b=30, l=40, r=20),
    showlegend=False,
)
fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)
fig.show()

## 5. How the Model Works

For each target quality, Lasso fits a penalized linear equation that automatically drops irrelevant features (coefficients → 0). The team style coefficients tell us how much a tactical context shift impacts each player quality. Pre-transfer qualities anchor the prediction — a good player stays good, but context adjusts the outcome.

## 6. Next Steps

**Phase 1 — Scale Data:** Expand from 5 same-league competitions (~6,700 transfers) to all leagues Twelve covers. More rows → better generalization and XGBoost becomes viable.

**Phase 2 — Cross-League Transfers:** Z-score team qualities against a global population instead of per-league. Use OUTCOME or ELO-like ratings as a league difficulty adjustment. Champions League matchups as a natural calibration bridge.

**Phase 3 — Nice to Have:**
- Player role clusters (pre/post) → "role change" as a feature (e.g., box-to-box → deep playmaker)
- Player-team quality gap — how good is the player relative to teammates (requires full squad data)
- Ensemble: Lasso for feature selection + XGBoost on surviving features